# 第06章 阈值处理（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 6 章跟随代码。**全量收录原书代码（例 6.1～例 6.13）**，缺失例题已按书补全，代码以网页笔记 docs/opencv/ch06.md 为准（含笔误修正与参数修正）。
> cv2.imshow / cv_show 需在本地 Python 环境运行（弹窗显示，Notebook 服务内看不到窗口）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg 等素材都在这里），否则 imread 返回 None。
> 素材替换：原书 lena.bmp、tiffany.bmp 用 picture1.jpg 以灰度方式读入顶替，computer.jpg 用 picture12.jpg 顶替；例 6.1、6.3、6.5、6.7、6.9 的数组用 np.random.randint 随机生成，每次运行输出都不同。

## 本章总览

阈值处理（thresholding）是指剔除图像内像素值高于一定值或者低于一定值的像素点——用一个（或一组）亮度门槛把像素分成两拨，直接产物是二值图，前景与背景就此分开。全章例题：例 6.1～例 6.10（threshold 五种类型）、例 6.11（自适应阈值）、例 6.12～例 6.13（Otsu 处理）。

- **threshold 函数**：`retval, dst = cv2.threshold(src, thresh, maxval, type)`，五种处理类型随意选；
- **自适应阈值处理**：明暗分布不均匀的图，交给 `cv2.adaptiveThreshold()` 按邻域逐点算阈值；
- **Otsu 处理**：不想手动挑阈值，叠加 `cv2.THRESH_OTSU` 遍历所有候选值自动挑类间方差最大者。

In [ ]:
import cv2

# cv_show 辅助函数：原书跟随代码沿用的显示封装（本地运行显示窗口用，弹窗显示后按任意键关闭）
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


## 6.1 threshold 函数

OpenCV 提供两个函数实现阈值处理：`cv2.threshold()`（固定阈值）与 `cv2.adaptiveThreshold()`（6.2 节）。先看前者，语法格式：

```python
retval, dst = cv2.threshold( src, thresh, maxval, type )
```

- **retval**：返回的阈值——普通类型下就是参数 thresh 的原样回传（如传 127 返回 127.0），只有叠加 Otsu / Triangle 标记时才是自动求出的最优阈值；
- **dst**：阈值分割结果图像，与原始图像具有相同的大小、类型；
- **src**：要进行阈值分割的图像，**可以是多通道的**（三通道各自独立与阈值比较），8 位或 32 位浮点型；
- **thresh**：要设定的阈值；
- **maxval**：type 为 THRESH_BINARY 或 THRESH_BINARY_INV 时需要设定的最大值——**只在这两种类型下生效**，TRUNC、TOZERO、TOZERO_INV 传别的 maxval 没有任何效果；
- **type**：阈值分割类型，"大于阈值"是**严格大于**（等于阈值的像素落在"其他情况"一侧）。

| 类型 | 处理规则（大于 thresh 的像素 / 其余像素） |
| --- | --- |
| cv2.THRESH_BINARY | 置为 maxval / 置为 0 |
| cv2.THRESH_BINARY_INV | 置为 0 / 置为 maxval |
| cv2.THRESH_TRUNC | 截断为 thresh / 保持不变 |
| cv2.THRESH_TOZERO_INV | 置为 0 / 保持不变 |
| cv2.THRESH_TOZERO | 保持不变 / 置为 0 |

另有三个标记值：`cv2.THRESH_OTSU`（值 8，叠加后用 Otsu 算法自动求阈值，见 6.3 节）、`cv2.THRESH_TRIANGLE`（值 16，三角法，适合单峰直方图）、`cv2.THRESH_MASK`（值 7，不是处理类型，是源码里剥离标记位用的位掩码，别当类型传入）。本节包含例 6.1～例 6.10。

### 6.1.1 二值化阈值处理（cv2.THRESH_BINARY）

### 例 6.1 · 对数组进行二值化阈值处理

二值化阈值处理（binary thresholding）把原始图像处理为仅有两个值的二值图像：大于阈值 thresh 的像素点置为最大值 maxval、其余置 0，即 `dst(x,y) = maxval if src(x,y) > thresh else 0`。返回值 t 打印出来是 127.0（浮点数）——传入的阈值原样带回。

In [ ]:
# ===== 6.1.1 例 6.1：数组上的 BINARY 阈值处理 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)  # 4 行 5 列随机灰度数组
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)      # 阈值 127、最大值 255、二值化
print("img=\n", img)     # 打印原始数组
print("t=", t)           # 返回的阈值：127.0
print("rst=\n", rst)     # 打印处理结果：只有 0 和 255

### 例 6.2 · 对图像进行二值化阈值处理

灰度读入（`cv2.imread("picture1.jpg", 0)`，原书 lena.bmp）后按 127 二值化：亮部（白色背景、浅色服饰）变白，暗部（头发、深色轮廓）变黑，全图只剩纯黑与纯白两个灰度级。

In [ ]:
# ===== 6.1.1 例 6.2：图像上的 BINARY 阈值处理 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                # 灰度读入（原书 lena.bmp）
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
cv2.imshow("img", img)
cv2.imshow("rst", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 6.1.2 反二值化阈值处理（cv2.THRESH_BINARY_INV）

### 例 6.3 · 对数组进行反二值化阈值处理

反二值化阈值处理（inverted binary thresholding）的结果也是仅有两个值的二值图像，与二值化阈值处理的区别在于二者对像素值的处理方式正好相反：大于阈值置 0、其余置 maxval——输出与 BINARY 黑白互换。

In [ ]:
# ===== 6.1.2 例 6.3：数组上的 BINARY_INV 阈值处理 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)  # 4 行 5 列随机灰度数组
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)  # 类型换成 BINARY_INV
print("img=\n", img)
print("t=", t)
print("rst=\n", rst)

### 例 6.4 · 对图像进行反二值化阈值处理

与例 6.2 的 BINARY 结果对照，黑白完全互换——亮部变黑、暗部变白，好比照片的"负片"。

In [ ]:
# ===== 6.1.2 例 6.4：图像上的 BINARY_INV 阈值处理 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                    # 灰度读入（原书 lena.bmp）
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
cv2.imshow("img", img)
cv2.imshow("rst", rst)
cv2.waitKey()
cv2.destroyAllWindows()

### 6.1.3 截断阈值化处理（cv2.THRESH_TRUNC）

### 例 6.5 · 对数组进行截断阈值化处理

截断阈值化处理（truncate thresholding）将大于阈值的像素点的值设定为阈值，小于或等于该阈值的像素点的值保持不变，即 `dst(x,y) = thresh if src(x,y) > thresh else src(x,y)`。注意：TRUNC 的结果不是二值图（输出里还有 thresh 和原始灰度两个层次），maxval 参数对它无效。

> **原书笔误**：原书表 6-1 的 THRESH_TRUNC 行与 6.1.3 节公式都把"其他情况"写成了 0——与它自己的正文"小于或等于该阈值的像素点的值保持不变"矛盾，例 6.5 的输出也证明小于阈值的值原样保留。上面公式已按正确规则更正。

In [ ]:
# ===== 6.1.3 例 6.5：数组上的 TRUNC 阈值处理 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)  # 4 行 5 列随机灰度数组
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)       # 类型为 TRUNC，maxval 无效
print("img=\n", img)
print("t=", t)
print("rst=\n", rst)

### 例 6.6 · 对图像进行截断阈值化处理

灰度图按 127 截断：右图亮部全部压成同一层灰（127），整体像蒙了一层灰纱——画面比原图"闷"，最亮的那批像素失去了层次。

In [ ]:
# ===== 6.1.3 例 6.6：图像上的 TRUNC 阈值处理 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                    # 灰度读入（原书 lena.bmp）
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)
cv2.imshow("img", img)
cv2.imshow("rst", rst)
cv2.waitKey()
cv2.destroyAllWindows()

**补充演示**（跟随代码原有）：彩色图也能做阈值处理——src 可以是多通道，三个通道各自独立与阈值比较。对 `picture3.jpg` 做 TRUNC（阈值 155）：B、G、R 三个通道里所有大于 155 的值都被截为 155，形状不变。常规用法仍是先转灰度再阈值，得到单通道掩模方便后续按位运算使用。

In [216]:
#===截断阈值化处理   cv2.THRESH_TRUNC===#
img=cv2.imread("picture3.jpg")
thresh=155
retval,dst=cv2.threshold(img,thresh,255,cv2.THRESH_TRUNC)
cv_show('dst',dst)

### 6.1.4 超阈值零处理（cv2.THRESH_TOZERO_INV）

### 例 6.7 · 对数组进行超阈值零处理

超阈值零处理（threshold to zero, inverted）将大于阈值的像素点的值处理为 0，小于或等于该阈值的像素点的值保持不变，即 `dst(x,y) = 0 if src(x,y) > thresh else src(x,y)`。注意"大于"是严格大于——等于阈值 127 的像素原样保留。

In [ ]:
# ===== 6.1.4 例 6.7：数组上的 TOZERO_INV 阈值处理 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)      # 4 行 5 列随机灰度数组
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)      # 大于阈值置 0
print("img=\n", img)
print("t=", t)
print("rst=\n", rst)

### 例 6.8 · 对图像进行超阈值零处理

亮部（大于 127）全部变黑，只留下暗部细节——保留的是原始灰度层次而非纯白。

In [ ]:
# ===== 6.1.4 例 6.8：图像上的 TOZERO_INV 阈值处理 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                        # 灰度读入（原书 lena.bmp）
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)
cv2.imshow("img", img)
cv2.imshow("rst", rst)
cv2.waitKey()
cv2.destroyAllWindows()

**补充演示**（跟随代码原有）：彩色图上的 TOZERO_INV——`picture3.jpg` 阈值 220，三个通道各自独立置零。

In [217]:
#===超阈值零处理    cv2.THRESH_TOZERO_INV===#
img=cv2.imread('picture3.jpg')

thresh=220
retval,dst=cv2.threshold(img,thresh,255,cv2.THRESH_TOZERO_INV)
cv_show('dst',dst)

### 6.1.5 低阈值零处理（cv2.THRESH_TOZERO）

### 例 6.9 · 对数组进行低阈值零处理

低阈值零处理（threshold to zero）将小于或等于阈值的像素点的值处理为 0，大于阈值的像素点的值保持不变——与 TOZERO_INV 正好相反，这次清零的是阈值以下的波谷。等于阈值 127 的像素同样归入被清零的一侧。保留下来的值原封不动，这是 TOZERO 与 BINARY 的本质区别：前者保留灰度层次，后者只有 0 和 maxval。

In [ ]:
# ===== 6.1.5 例 6.9：数组上的 TOZERO 阈值处理 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[4, 5], dtype=np.uint8)      # 4 行 5 列随机灰度数组
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)          # 小于等于阈值置 0
print("img=\n", img)
print("t=", t)
print("rst=\n", rst)

### 例 6.10 · 对图像进行低阈值零处理

暗部整体变黑、亮部保留原有灰度——相当于把背景"压黑"、前景"浮出来"，常用于突出高亮区域。

In [ ]:
# ===== 6.1.5 例 6.10：图像上的 TOZERO 阈值处理 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                        # 灰度读入（原书 lena.bmp）
t, rst = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)
cv2.imshow("img", img)
cv2.imshow("rst", rst)
cv2.waitKey()
cv2.destroyAllWindows()

**补充演示**（跟随代码原有）：彩色图上的 TOZERO——`picture3.jpg` 阈值 100。

In [218]:
#===低阈值零处理
img=cv2.imread('picture3.jpg')

thresh=100
retval,dst=cv2.threshold(img,thresh,255,cv2.THRESH_TOZERO)
cv_show('dst',dst)

## 6.2 自适应阈值处理

色彩均衡的图像直接使用一个阈值就能完成阈值化处理；但图像明暗分布不均匀时，只使用一个阈值就无法得到清晰有效的分割结果。改进思路是用变化的阈值：自适应阈值处理（adaptive thresholding）逐个像素计算其周围临近区域的加权平均值来获得该像素**自己的阈值**，再用这个阈值处理当前像素，能够更好地处理明暗差异较大的图像。

```python
dst = cv2.adaptiveThreshold( src, maxValue, adaptiveMethod, thresholdType, blockSize, C )
```

- **src**：必须是 **8 位单通道**图像（彩色图先 `cv2.cvtColor` 转灰度）；
- **maxValue**：最大值；
- **adaptiveMethod**：自适应方法，有两种——`cv2.ADAPTIVE_THRESH_MEAN_C`（邻域内所有像素点权重一致，即求平均）与 `cv2.ADAPTIVE_THRESH_GAUSSIAN_C`（权重通过高斯方程得到，离中心越近权重越大）；自适应阈值 = 每个像素 blockSize 邻域的加权平均值 − 常量 C；
- **thresholdType**：阈值处理方式，只能是 THRESH_BINARY 或 THRESH_BINARY_INV 之一；
- **blockSize**：块大小，计算一个像素的阈值时所使用的邻域尺寸，必须是**奇数**（3、5、7 等）；
- **C**：常量。C 越大，阈值（均值 − C）越低，THRESH_BINARY 下被判白的像素越多。

### 例 6.11 · 全局阈值 vs 自适应阈值

对 `picture12.jpg`（原书 computer.jpg 计算器图，明暗差异大）分别用 `cv2.threshold`（127）与两种 `adaptiveThreshold`（块 5、C=3）处理：全局 127 后暗部细节大量丢失，而两种自适应方法都把纹理完整勾了出来——自适应阈值处理保留了更多的细节信息。

In [ ]:
# ===== 6.2 例 6.11：全局阈值 vs 自适应阈值 =====
import cv2

img = cv2.imread("picture12.jpg", 0)          # 灰度读入（原书 computer.jpg 计算器图）
t1, thd = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)          # 全局固定阈值 127
athdMEAN = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                 cv2.THRESH_BINARY, 5, 3)          # 邻域均值法，块 5，C=3
athdGAUS = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 5, 3)          # 高斯加权法，块 5，C=3
cv2.imshow("img", img)
cv2.imshow("thd", thd)
cv2.imshow("athdMEAN", athdMEAN)
cv2.imshow("athdGAUS", athdGAUS)
cv2.waitKey()
cv2.destroyAllWindows()

## 6.3 Otsu 处理

使用 `cv2.threshold()` 需要自定义阈值；图像灰度级的分布不均衡时，127 这个"标准答案"可能毫无区分度——例如一幅只有 123 和 126 两种灰度的图，以 127 分割全图变纯黑。Otsu 方法（**Otsu method**，也称大津法、最大类间方差法）能够根据当前图像给出**最佳的类间分割阈值**：遍历所有可能阈值，找到最佳的阈值。在 OpenCV 中给 type 多叠加一个 `cv2.THRESH_OTSU` 即可：

```python
t, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
```

- type 用加号叠加 `cv2.THRESH_OTSU`；设定的阈值写 0（传什么都会被忽略，写 0 是约定俗成）；
- 返回值 t 是 Otsu 方法计算得到并使用的最优阈值——遍历 0~255 每个候选阈值，取**类间方差** σ² = w0·w1·(μ0−μ1)² 最大者（w0、w1 为两类像素占比，μ0、μ1 为两类平均灰度）；
- Otsu 只支持 8 位单通道图，彩色图或 float32 图叠加 THRESH_OTSU 会直接报错。

### 例 6.12 · 测试 Otsu 阈值处理的实现

在 123/126 图上对比：普通 127 阈值全军覆没（全 0）；Otsu 自动找出最优阈值并分割成功，右下 3×3 的 126 变为 255，与"以 125 分割"的手工结果一模一样（实测 t2 = 123.0）。另有三角法标记 `cv2.THRESH_TRIANGLE`（值 16）：对直方图找最高峰，取峰点与直方图远端连线距离最大的灰度级作阈值，适合**单峰直方图**的图，用法与 OTSU 完全一致（加号叠加）——实测例 6.12 的图上三角法返回 125.0，恰好是原书手工挑选的那个值。

In [ ]:
# ===== 6.3 例 6.12：123/126 图上对比普通阈值与 Otsu =====
import cv2
import numpy as np

img = np.zeros((5, 5), dtype=np.uint8)
img[0:6, 0:6] = 123        # 越界切片：5×5 数组上等效 img[0:5, 0:5]，全图置 123
img[2:6, 2:6] = 126        # 越界切片：等效 img[2:5, 2:5]，右下 3×3 置 126
print("img=\n", img)
t1, thd = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)                # 普通阈值 127
print("thd=\n", thd)
t2, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # Otsu 自动阈值
print("otsu=\n", otsu)
print("t2=", t2)           # Otsu 找到的最优阈值

### 例 6.13 · 真实图上的 127 阈值 vs Otsu 阈值

对 `picture1.jpg`（原书 tiffany.bmp 亮调人像，整体偏亮）分别做普通二值化（127）与 Otsu：127 阈值下得到大量白色区域、浅色背景与服饰的层次糊成一片白（实测白像素约占 80%）；Otsu 自动采用最优阈值（实测 160），大片中灰的背景被压回黑侧，人物轮廓重新分明，得到较好的处理结果。

In [ ]:
# ===== 6.3 例 6.13：真实图上的 127 阈值 vs Otsu 阈值 =====
import cv2

img = cv2.imread("picture1.jpg", 0)           # 灰度读入（原书 tiffany.bmp 亮调人像）
t1, thd = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)                  # 普通二值化，127
t2, otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # Otsu 自动阈值
cv2.imshow("img", img)
cv2.imshow("thd", thd)
cv2.imshow("otsu", otsu)
cv2.waitKey()
cv2.destroyAllWindows()

## 坑与陷阱（速查）

1. "大于阈值"是**严格大于**——等于阈值的像素永远落在"其他情况"一侧（BINARY@127 时值为 127 的像素变 0，实测 `[126,127,128] → [0,0,255]`）；
2. 原书 TRUNC 公式有误："其他情况"应保持不变（src(x,y)），且 TRUNC 输出不是二值图；
3. maxval 只对 BINARY / BINARY_INV 生效，其余类型传任意 maxval 都无效果；
4. adaptiveThreshold 三道门槛：src 必须 8 位单通道（彩色图先 cvtColor）、thresholdType 只能 BINARY/BINARY_INV、blockSize 必须是奇数；Otsu 同样只支持 8 位单通道；
5. 普通阈值的 retval 不是"算出来的"，就是传入 thresh 的原样回传；OTSU 模式下传入的 thresh 被忽略，"阈值写 0"只是惯例；
6. NumPy 越界切片静默截断：例 6.12 的 `img[0:6, 0:6]` 作用在 5×5 数组上不报错，等效 `img[0:5, 0:5]`；
7. 随机数组例题（例 6.1～6.9）每次运行输出都不同，对照书上输出逐位核对是白费劲——拿固定小数组验证规则才可靠。